# Splatfacto (Nerfstudio) 10k on Colab

This notebook trains Splatfacto for 10,000 iterations on `gerrard-hall` and writes the same metrics used by the Inria and Mip notebooks:

- PSNR, SSIM, LPIPS
- Training time
- Peak GPU memory
- Number of Gaussians
- PLY file size

Run on a Colab GPU runtime (T4/L4/A100), with dataset and outputs on Google Drive.

In [ ]:
from pathlib import Path
import csv
import json
import os
import re
import shutil
import subprocess
import threading
import time

SCENE_NAME = "gerrard-hall"
MAX_ITERATIONS = 10_000

DRIVE_ROOT = Path("/content/drive/MyDrive/worldmodeling")
RUNS_ROOT = DRIVE_ROOT / "sota_runs"
MODEL_ROOT = RUNS_ROOT / "splatfacto" / SCENE_NAME
METRICS_DIR = RUNS_ROOT / "metrics"

WORK_ROOT = Path("/content/worldmodeling_splatfacto")
NS_DATA_DIR = WORK_ROOT / "ns_data" / SCENE_NAME

MODEL_ROOT.mkdir(parents=True, exist_ok=True)
METRICS_DIR.mkdir(parents=True, exist_ok=True)
WORK_ROOT.mkdir(parents=True, exist_ok=True)

print("Scene:", SCENE_NAME)
print("Iterations:", MAX_ITERATIONS)
print("Model root:", MODEL_ROOT)
print("Metrics dir:", METRICS_DIR)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

scene_candidates = [
    DRIVE_ROOT / "scenes" / SCENE_NAME,
    DRIVE_ROOT / "scenes" / SCENE_NAME / SCENE_NAME,
]

SCENE_DIR = None
for cand in scene_candidates:
    if (cand / "images").exists() and (cand / "sparse").exists():
        SCENE_DIR = cand
        break

if SCENE_DIR is None:
    raise FileNotFoundError(
        "Could not find scene folder with images/ and sparse/. Checked:\n" + "\n".join(map(str, scene_candidates))
    )

IMAGE_DIR = SCENE_DIR / "images"
SPARSE_DIR = SCENE_DIR / "sparse"

print("Using scene:", SCENE_DIR)
print("Images:", len(list(IMAGE_DIR.glob('*'))))

In [ ]:
import torch

print("PyTorch:", torch.__version__)
if not torch.cuda.is_available():
    raise RuntimeError("No CUDA GPU detected. Enable GPU runtime in Colab.")

gpu = torch.cuda.get_device_properties(0)
print("GPU:", gpu.name)
print(f"Total VRAM: {gpu.total_memory / 1e9:.2f} GB")

In [ ]:
def run_cmd(cmd, cwd=None, check=True, capture=True):
    print(f"\\n$ {cmd}")
    result = subprocess.run(
        cmd,
        shell=True,
        cwd=str(cwd) if cwd else None,
        text=True,
        capture_output=capture,
    )
    if capture:
        if result.stdout:
            print(result.stdout[-4000:])
        if result.stderr:
            print(result.stderr[-4000:])
    if check and result.returncode != 0:
        raise RuntimeError(f"Command failed ({result.returncode}): {cmd}")
    return result

def build_gpu_monitor(interval_sec=2.0):
    state = {"max_mem_mb": 0}
    stop_event = threading.Event()

    def _poll():
        while not stop_event.is_set():
            try:
                out = subprocess.check_output(
                    "nvidia-smi --query-gpu=memory.used --format=csv,noheader,nounits",
                    shell=True,
                    text=True,
                ).strip()
                used = int(out.splitlines()[0])
                state["max_mem_mb"] = max(state["max_mem_mb"], used)
            except Exception:
                pass
            time.sleep(interval_sec)

    thread = threading.Thread(target=_poll, daemon=True)
    return state, stop_event, thread

def count_vertices_from_ply(ply_path):
    with open(ply_path, "r", encoding="utf-8", errors="ignore") as f:
        for line in f:
            line = line.strip()
            if line.startswith("element vertex"):
                return int(line.split()[-1])
            if line == "end_header":
                break
    return None

def extract_scalar(obj, names):
    if not isinstance(obj, dict):
        return None
    lowered = {str(k).lower(): v for k, v in obj.items()}
    for n in names:
        if n.lower() in lowered and isinstance(lowered[n.lower()], (int, float)):
            return float(lowered[n.lower()])
    return None

In [ ]:
# Install Nerfstudio and dependencies
run_cmd("pip install -q ninja")
run_cmd("pip install -q nerfstudio")

# tiny-cuda-nn is commonly needed for Nerfstudio training
tcnn_check = run_cmd("python -c \"import tinycudann\"", check=False)
if tcnn_check.returncode != 0:
    run_cmd("pip install -q git+https://github.com/NVlabs/tiny-cuda-nn/#subdirectory=bindings/torch")

# Verify CLIs
for cli in ["ns-process-data", "ns-train", "ns-export", "ns-eval"]:
    ok = run_cmd(f"{cli} --help > /dev/null 2>&1", check=False)
    print(f"{cli}:", "OK" if ok.returncode == 0 else "MISSING")

In [ ]:
# Ensure COLMAP model path is in sparse/0 and in BIN format
run_cmd("apt-get -qq update")
run_cmd("apt-get -qq install -y colmap")

SPARSE_0 = SPARSE_DIR / "0"
if not SPARSE_0.exists():
    SPARSE_0.mkdir(parents=True, exist_ok=True)
    for fn in ["cameras.txt", "images.txt", "points3D.txt", "cameras.bin", "images.bin", "points3D.bin", "points3D.ply"]:
        src = SPARSE_DIR / fn
        if src.exists():
            shutil.copy2(src, SPARSE_0 / fn)

if not (SPARSE_0 / "cameras.bin").exists() and (SPARSE_0 / "cameras.txt").exists():
    run_cmd(f"colmap model_converter --input_path '{SPARSE_0}' --output_path '{SPARSE_0}' --output_type BIN")

if not (SPARSE_0 / "cameras.bin").exists():
    raise FileNotFoundError(f"No COLMAP cameras.bin found after conversion at {SPARSE_0}")

print("Using COLMAP model:", SPARSE_0)
run_cmd(f"ls -lah '{SPARSE_0}'")

In [ ]:
# Build Nerfstudio transforms from existing COLMAP poses
if NS_DATA_DIR.exists():
    shutil.rmtree(NS_DATA_DIR)
NS_DATA_DIR.mkdir(parents=True, exist_ok=True)

process_cmd = (
    f"ns-process-data images "
    f"--data '{IMAGE_DIR}' "
    f"--output-dir '{NS_DATA_DIR}' "
    f"--skip-colmap "
    f"--colmap-model-path '{SPARSE_0}'"
)
run_cmd(process_cmd)

transforms_path = NS_DATA_DIR / "transforms.json"
if not transforms_path.exists():
    raise FileNotFoundError(f"Expected transforms.json not found: {transforms_path}")

print("Transforms ready:", transforms_path)

In [ ]:
# Train Splatfacto 10k
train_cmd = (
    f"ns-train splatfacto "
    f"--data '{NS_DATA_DIR}' "
    f"--output-dir '{MODEL_ROOT}' "
    f"--max-num-iterations {MAX_ITERATIONS} "
    f"--viewer.quit-on-train-completion True"
)

gpu_state, gpu_stop, gpu_thread = build_gpu_monitor()
gpu_thread.start()

t0 = time.time()
run_cmd(train_cmd, capture=False)
training_time_sec = time.time() - t0

gpu_stop.set()
gpu_thread.join(timeout=5)

peak_gpu_gb = gpu_state["max_mem_mb"] / 1024.0
print(f"Training time: {training_time_sec/60:.2f} minutes")
print(f"Peak GPU memory: {peak_gpu_gb:.2f} GB")

configs = sorted(MODEL_ROOT.glob("**/config.yml"), key=lambda p: p.stat().st_mtime)
if not configs:
    raise FileNotFoundError(f"No config.yml found under {MODEL_ROOT}")
CONFIG_PATH = configs[-1]
print("Using config:", CONFIG_PATH)

In [ ]:
# Evaluate PSNR/SSIM/LPIPS
EVAL_JSON = MODEL_ROOT / "eval_splatfacto_10k.json"
eval_cmd = f"ns-eval --load-config '{CONFIG_PATH}' --output-path '{EVAL_JSON}'"
run_cmd(eval_cmd)

if not EVAL_JSON.exists():
    raise FileNotFoundError(f"Expected eval output not found: {EVAL_JSON}")

with open(EVAL_JSON, "r") as f:
    eval_obj = json.load(f)

flat_candidates = []
if isinstance(eval_obj, dict):
    flat_candidates.append(eval_obj)
    for v in eval_obj.values():
        if isinstance(v, dict):
            flat_candidates.append(v)

psnr = ssim = lpips = None
for cand in flat_candidates:
    if psnr is None:
        psnr = extract_scalar(cand, ["psnr", "test_psnr", "results/psnr"])
    if ssim is None:
        ssim = extract_scalar(cand, ["ssim", "test_ssim", "results/ssim"])
    if lpips is None:
        lpips = extract_scalar(cand, ["lpips", "test_lpips", "results/lpips"])

print("PSNR:", psnr)
print("SSIM:", ssim)
print("LPIPS:", lpips)

In [ ]:
# Export PLY and gather storage metrics
EXPORT_DIR = MODEL_ROOT / "exports" / "splatfacto_10k"
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

export_script = f"""
import torch
_orig_load = torch.load
def patched_load(*args, **kwargs):
    kwargs.setdefault('weights_only', False)
    return _orig_load(*args, **kwargs)
torch.load = patched_load

import sys
sys.argv = [
    'ns-export', 'gaussian-splat',
    '--load-config', '{CONFIG_PATH}',
    '--output-dir', '{EXPORT_DIR}'
]
from nerfstudio.scripts.exporter import entrypoint
entrypoint()
"""

run_cmd(f'python -c "{export_script}"')

ply_files = sorted(EXPORT_DIR.glob("*.ply"), key=lambda p: p.stat().st_mtime)
if not ply_files:
    raise FileNotFoundError(f"No exported PLY found in {EXPORT_DIR}")
PLY_PATH = ply_files[-1]

num_gaussians = count_vertices_from_ply(PLY_PATH)
ply_size_mb = PLY_PATH.stat().st_size / 1e6

print("PLY:", PLY_PATH)
print("#Gaussians:", num_gaussians)
print(f"PLY size: {ply_size_mb:.3f} MB")

In [ ]:
# Save final summary with the same schema as the other two notebooks
summary = {
    "model": "Splatfacto",
    "scene": SCENE_NAME,
    "iterations": MAX_ITERATIONS,
    "PSNR": psnr,
    "SSIM": ssim,
    "LPIPS": lpips,
    "training_time_sec": round(training_time_sec, 3),
    "training_time_min": round(training_time_sec / 60.0, 3),
    "peak_gpu_gb": round(peak_gpu_gb, 3),
    "num_gaussians": num_gaussians,
    "ply_file_size_mb": round(ply_size_mb, 3),
    "ply_path": str(PLY_PATH),
    "raw_eval_json": eval_obj,
}

json_path = METRICS_DIR / f"splatfacto_{SCENE_NAME}_10k.json"
csv_path = METRICS_DIR / "sota_metrics_10k.csv"

with open(json_path, "w") as f:
    json.dump(summary, f, indent=2)

headers = [
    "model", "scene", "iterations", "PSNR", "SSIM", "LPIPS",
    "training_time_sec", "training_time_min", "peak_gpu_gb",
    "num_gaussians", "ply_file_size_mb", "ply_path",
]
write_header = not csv_path.exists()
with open(csv_path, "a", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=headers)
    if write_header:
        writer.writeheader()
    writer.writerow({k: summary.get(k) for k in headers})

print("Saved JSON:", json_path)
print("Updated CSV:", csv_path)
print(json.dumps(summary, indent=2))

## Runtime Notes

- All long-lived outputs are written to Google Drive under `worldmodeling/sota_runs`.
- If Colab disconnects, rerun from the top of this notebook in a fresh runtime.
- Metrics are read from `ns-eval` JSON to avoid brittle stdout regex parsing.